# 🚀 GCP Vault: Supercharged Colab GPU Render & AI Caption Engine
Notebook ini adalah **Tier 1 (Ultimate Optimization)** dari Kerangka Kerja Render Bertingkat Faisal.

### Keunggulan Utama:
1. **Biaya Rp 0 (100% Gratis)**: Menggunakan alokasi GPU NVIDIA Tesla T4 (16 GB VRAM) Google Colab tanpa memotong sepeser pun saldo promo GCP.
2. **Akselerasi Hardware NVENC**: Render video Full HD 60fps dengan codec `h264_nvenc` atau `hevc_nvenc` (kecepatan hingga 5x lebih cepat dari CPU laptop).
3. **AI Auto-Subtitle (Faster-Whisper CUDA)**: Otomatis mendengarkan audio narasi bahasa Indonesia, membuat subtitle `.srt` presisi, dan menempelkannya dengan format font caption vertikal TikTok/Reels.
4. **Sinkronisasi Otomatis Cloud Storage**: Membaca antrean dari GCS `gs://project-a452c4e5-ac83-423c-afd-vault-render/input/` dan otomatis mengunggah video hasil ke `output/`.
5. **Notifikasi Otomatis Webhook**: Mengirim laporan selesai ke Always-On VM Jakarta saat render tuntas.

--- 
## Tahap 1: Inisialisasi Lingkungan & Driver GPU Tesla T4

In [ ]:
# Periksa status kartu grafis dan CUDA
!nvidia-smi

# Instal FFmpeg, paket subtitle, dan AI Faster-Whisper berbasis CUDA
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core fonts-liberation
!pip install -q faster-whisper requests google-cloud-storage

--- 
## Tahap 2: Konfigurasi Project & Autentikasi GCS (Opsional / Otomatis)

In [ ]:
import os
import sys
import time
import requests
from pathlib import Path

# Parameter Ekosistem GCP Vault
PROJECT_ID = "project-a452c4e5-ac83-423c-afd"
BUCKET_NAME = f"{PROJECT_ID}-vault-render"
JAKARTA_RUNNER_URL = "http://34.101.240.185:443/alert"

# Buat folder kerja lokal Colab
LOCAL_INPUT = Path("/content/input")
LOCAL_OUTPUT = Path("/content/output")
LOCAL_INPUT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

# Hubungkan ke Google Cloud (Pilih akun Google Anda yang memiliki akses GCP)
try:
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Autentikasi Google Cloud Storage: BERHASIL ✓")
    !gcloud config set project {PROJECT_ID} --quiet
    # Sinkronkan berkas input dari GCS
    !gcloud storage rsync -r gs://{BUCKET_NAME}/input/ /content/input/
except Exception as e:
    print(f"Mode mandiri lokal aktif (GCS auth dilewati/manual upload): {e}")

--- 
## Tahap 3: Mesin AI Transkripsi & Generator Subtitle Otomatis (Faster-Whisper CUDA)
Jika Anda memiliki berkas video atau audio narasi di folder `/content/input`, modul ini otomatis menghasilkan subtitle presisi tinggi dalam Bahasa Indonesia.

In [ ]:
def generate_subtitles(audio_path, srt_path):
    """Transkripsi audio/video ke subtitle .srt menggunakan GPU CUDA T4."""
    from faster_whisper import WhisperModel
    print(f"[AI WHISPER] Memuat model Whisper ke GPU VRAM: {audio_path}...")
    
    # Gunakan model 'small' atau 'medium' untuk akurasi tinggi bahasa Indonesia
    model = WhisperModel("small", device="cuda", compute_type="float16")
    segments, info = model.transcribe(str(audio_path), language="id", beam_size=5)
    
    print(f"[AI WHISPER] Bahasa terdeteksi: {info.language} (Akurasi: {info.language_probability:.2f})")
    
    def format_timestamp(seconds):
        hrs = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 1000)
        return f"{hrs:02d}:{mins:02d}:{secs:02d},{millis:03d}"
    
    with open(srt_path, "w", encoding="utf-8") as f:
        for idx, segment in enumerate(segments, start=1):
            start_ts = format_timestamp(segment.start)
            end_ts = format_timestamp(segment.end)
            text = segment.text.strip()
            f.write(f"{idx}\n{start_ts} --> {end_ts}\n{text}\n\n")
            
    print(f"[AI WHISPER] Subtitle berhasil dibuat: {srt_path} ✓")
    return True

print("Modul AI Subtitle Generator siap digunakan.")

--- 
## Tahap 4: Eksekusi Render Video Berakselerasi Hardware GPU NVENC (1080x1920 60fps)

In [ ]:
# Periksa apakah ada video/audio kustom di folder input
input_files = list(LOCAL_INPUT.glob("*"))
media_files = [f for f in input_files if f.suffix.lower() in [".mp4", ".mov", ".avi", ".mkv", ".wav", ".mp3"]]

output_video_path = LOCAL_OUTPUT / "supercharged_colab_render.mp4"
start_time = time.time()

if media_files:
    target_file = media_files[0]
    print(f"[RENDER] Memproses berkas media: {target_file.name}")
    srt_file = LOCAL_OUTPUT / "subtitles.srt"
    
    # Generate subtitle otomatis dengan AI GPU
    try:
        generate_subtitles(target_file, srt_file)
        subtitle_filter = f",subtitles='{srt_file}':force_style='FontSize=16,FontName=Liberation Sans,PrimaryColour=&H00FFFF,BackColour=&H80000000,BorderStyle=4'"
    except Exception as e:
        print(f"Gagal generate subtitle otomatis, render tanpa subtitle: {e}")
        subtitle_filter = ""
        
    # Render dengan GPU NVENC
    cmd = f"""ffmpeg -y -hwaccel cuda -i "{target_file}" \
        -vf "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920{subtitle_filter}" \
        -c:v h264_nvenc -preset p5 -tune hq -b:v 6M -pix_fmt yuv420p \
        -c:a aac -b:a 192k \
        "{output_video_path}""""
    !{cmd}
else:
    print("[DEMO BENCHMARK] Tidak ada berkas di input, menjalankan Demo Sintesis 1080x1920 60fps GPU...")
    !ffmpeg -y \
        -f lavfi -i testsrc=duration=15:size=1080x1920:rate=60 \
        -f lavfi -i sine=frequency=440:duration=15 \
        -vf "drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf:text='SUPERCHARGED COLAB GPU T4':fontcolor=white:fontsize=48:x=(w-text_w)/2:y=(h-text_h)/2-120:box=1:boxcolor=black@0.6:boxborderw=10, drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf:text='NVIDIA NVENC Hardware Fast Render':fontcolor=yellow:fontsize=32:x=(w-text_w)/2:y=(h-text_h)/2:box=1:boxcolor=black@0.6:boxborderw=10, drawtext=fontfile=/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf:text='Biaya Tambahan Rp 0':fontcolor=lime:fontsize=36:x=(w-text_w)/2:y=(h-text_h)/2+80:box=1:boxcolor=black@0.6:boxborderw=10" \
        -c:v h264_nvenc -preset p5 -tune hq -b:v 6M -pix_fmt yuv420p \
        -c:a aac -b:a 192k \
        {output_video_path}

elapsed = time.time() - start_time
print(f"\n==========================================================")
print(f"   RENDER SELESAI DALAM {elapsed:.2f} DETIK VIA NVIDIA T4!")
print(f"==========================================================")
!ls -lh /content/output

--- 
## Tahap 5: Sinkronisasi Balik ke GCS & Kirim Notifikasi Webhook Jakarta

In [ ]:
# 1. Unggah hasil render ke Cloud Storage
try:
    !gcloud storage rsync -r /content/output/ gs://{BUCKET_NAME}/output/
    print("Hasil render berhasil diunggah ke Google Cloud Storage ✓")
except Exception as e:
    print(f"Gagal upload ke GCS (bisa unduh manual): {e}")

# 2. Kirim Notifikasi Webhook ke VM Jakarta Always-On
try:
    payload = {
        "source": "google-colab-gpu",
        "title": "Render Colab GPU T4 Selesai",
        "message": f"Video berhasil dirender via GPU Tesla T4 dalam {elapsed:.2f} detik. Berkas telah tersedia di Render_Vault/output.",
        "data": {"status": "SUCCESS", "duration_sec": round(elapsed, 2)}
    }
    headers = {"Authorization": "Bearer VaultSecret2026"}
    resp = requests.post(JAKARTA_RUNNER_URL, json=payload, headers=headers, timeout=5)
    print(f"Notifikasi ke Always-On Runner Jakarta: {resp.status_code} {resp.reason} ✓")
except Exception as e:
    print(f"Notifikasi webhook dilewati: {e}")

# 3. Unduh berkas langsung ke browser laptop (Cadangan lokal instan)
from google.colab import files
if output_video_path.exists():
    print("Menyiapkan download berkas ke laptop...")
    files.download(str(output_video_path))
    print("Selesai!")